# T1 · Big Data e Ingeniería de Datos

**Estudiante:** Juan Pablo Castro  
**Fuente:** Precipitación del IDEAM (`s54a-sgyg`)  
**Equipo de la medición definitiva:** MacBook Neo · macOS 27.0 · `arm64`

Este cuaderno realiza una medición acotada y reproducible sobre un día completo, compara el día inmediatamente anterior y conserva todas las salidas necesarias para la ficha técnica. No descarga el conjunto histórico completo.

Las cifras definitivas se midieron en macOS el 29 de julio de 2026 y sustituyen a la ejecución previa en Windows. El período, el archivo, la clave candidata, la licencia y el método de estimación de \(g\) no cambian. Si el CSV del 22 de junio de 2026 ya está en `evidencia/` y su conteo coincide con el de la API, el cuaderno lo reutiliza en lugar de volver a descargarlo, para que \(S_0\) no se altere.


## 1. Importaciones y entorno

In [1]:
import csv
import hashlib
import io
import json
import math
import os
import platform
import sys
import time
from datetime import datetime
from importlib import metadata as importlib_metadata
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import psutil
import requests
from IPython.display import display

LOCAL_TZ = ZoneInfo("America/Bogota")
EXECUTION_START = datetime.now(LOCAL_TZ)
BYTES_PER_MB = 1024 ** 2
BYTES_PER_GB = 1024 ** 3

environment = {
    "python": sys.version.replace("\n", " "),
    "sistema_operativo": platform.platform(),
    "pandas": pd.__version__,
    "psutil": importlib_metadata.version("psutil"),
    "requests": importlib_metadata.version("requests"),
    "fecha_hora_local": EXECUTION_START.isoformat(),
    "ruta_trabajo": str(Path.cwd().resolve()),
    "conversion_bytes_GB": "bytes / (1024 ** 3)",
}
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "python": "3.12.8 (main, Jun 29 2026, 10:05:04) [Clang 21.0.0 (clang-2100.1.1.101)]",
  "sistema_operativo": "macOS-27.0-arm64-arm-64bit",
  "pandas": "2.2.3",
  "psutil": "7.2.2",
  "requests": "2.32.5",
  "fecha_hora_local": "2026-07-29T15:17:07.880500-05:00",
  "ruta_trabajo": "/Users/juanpablocastro/Downloads/T1_Castro_JuanPablo",
  "conversion_bytes_GB": "bytes / (1024 ** 3)"
}


## 2. Rutas y parámetros reproducibles

In [2]:
DATASET_ID = "s54a-sgyg"
PORTAL_URL = "https://www.datos.gov.co/Ambiente-y-Desarrollo-Sostenible/Precipitaci-n/s54a-sgyg"
API_CSV = f"https://www.datos.gov.co/resource/{DATASET_ID}.csv"
API_JSON = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"
METADATA_URL = f"https://www.datos.gov.co/api/views/{DATASET_ID}"

cwd = Path.cwd().resolve()
PROJECT_DIR = cwd if cwd.name == "T1_Castro_JuanPablo" else cwd / "T1_Castro_JuanPablo"
EVIDENCE_DIR = PROJECT_DIR / "evidencia"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

# Dos días completos consecutivos, publicados y de igual duración.
PERIOD_A_START = "2026-06-21T00:00:00"
PERIOD_A_END = "2026-06-22T00:00:00"
PERIOD_B_START = "2026-06-22T00:00:00"
PERIOD_B_END = "2026-06-23T00:00:00"
MAIN_PATH = EVIDENCE_DIR / "precipitacion_2026-06-22.csv"
COMPARISON_PATH = EVIDENCE_DIR / "precipitacion_2026-06-21.csv"
ORDER = "fechaobservacion,codigoestacion,codigosensor,:id"
PAGE_SIZE = 50_000

session = requests.Session()
session.headers.update({
    "User-Agent": "T1-Castro-JuanPablo-academic/1.0",
    "Accept-Encoding": "identity",
})

print("Carpeta de entrega:", PROJECT_DIR)
print("Período principal:", PERIOD_B_START, "a", PERIOD_B_END, "(fin exclusivo)")
print("Período comparable:", PERIOD_A_START, "a", PERIOD_A_END, "(fin exclusivo)")

Carpeta de entrega: /Users/juanpablocastro/Downloads/T1_Castro_JuanPablo
Período principal: 2026-06-22T00:00:00 a 2026-06-23T00:00:00 (fin exclusivo)
Período comparable: 2026-06-21T00:00:00 a 2026-06-22T00:00:00 (fin exclusivo)


## 3. Metadatos oficiales y nombres reales de columnas

In [3]:
def get_retry(url, params=None, attempts=4, timeout=180):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            last_error = exc
            if attempt == attempts:
                break
            wait = 2 ** (attempt - 1)
            print(f"Reintento {attempt}: espera de {wait} s.")
            time.sleep(wait)
    raise RuntimeError(f"No fue posible consultar {url}") from last_error


metadata_response = get_retry(METADATA_URL)
(EVIDENCE_DIR / "metadata_dataset.json").write_bytes(metadata_response.content)
metadata = metadata_response.json()
metadata_columns = pd.DataFrame([
    {
        "nombre_visible": column.get("name"),
        "nombre_api": column.get("fieldName"),
        "tipo_declarado": column.get("dataTypeName"),
        "descripcion": column.get("description"),
    }
    for column in metadata.get("columns", [])
])
custom_data = (
    metadata.get("metadata", {})
    .get("custom_fields", {})
    .get("Información de Datos", {})
)
metadata_summary = {
    "entidad": metadata.get("attribution"),
    "descripcion": metadata.get("description"),
    "licencia": metadata.get("license"),
    "frecuencia_actualizacion": custom_data.get("Frecuencia de Actualización"),
}
print(json.dumps(metadata_summary, ensure_ascii=False, indent=2))
display(metadata_columns)

discovery_response = get_retry(API_CSV, {"$limit": 10})
discovery_df = pd.read_csv(io.BytesIO(discovery_response.content), low_memory=False)
print("Consulta de descubrimiento:", discovery_response.url)
print("Columnas reales:", discovery_df.columns.tolist())
print("Tipos inferidos:")
print(discovery_df.dtypes.to_string())
display(discovery_df.head())

{
  "entidad": "Instituto de Hidrología, Meteorología y Estudios Ambientales - IDEAM, Bogotá D.C.",
  "descripcion": "Contiene datos sobre la cantidad de lluvia registrada cada 10 minutos en diferentes estaciones meteorológicas ubicadas sobre el territorio colombiano. Los cuales han sido verificados mediante un control de calidad básico, de acuerdo con las recomendaciones de la Organización Meteorológica Mundial (OMM). \nPor lo anterior, se han corregido o eliminado valores que no han cumplido con estos controles y podrían requerir validaciones adicionales según el uso específico. \nEste conjunto de datos puede ser útil para analizar el comportamiento de la precipitación, determinar fenómenos extremos, realizar estudios climáticos e hidrológicos o apoyar sistemas de alerta temprana. \nSe recomienda a los usuarios aplicar criterios adicionales de verificación para usos sensibles o técnicos especializados.\nLos datos disponibles a través de este medio pueden ser libremente consumidos baj

,nombre_visible,nombre_api,tipo_declarado,descripcion
0,CodigoEstacion,codigoestacion,text,Corresponde al valor de identificación de la e...
1,CodigoSensor,codigosensor,text,Código de identificación asignado al sensor
2,FechaObservacion,fechaobservacion,calendar_date,Fecha en la cual se realiza la medición
3,ValorObservado,valorobservado,number,Valor medido
4,NombreEstacion,nombreestacion,text,Corresponde a la identificación de la estación...
5,Departamento,departamento,text,Nombre del departamento donde se ubica la esta...
6,Municipio,municipio,text,Nombre del Municipio donde se localiza la esta...
7,ZonaHidrografica,zonahidrografica,text,Zona hidrográfica sobre la cual está ubicada l...
8,Latitud,latitud,number,Corresponde a la latitud en la cual se ubica l...
9,Longitud,longitud,number,Corresponde a la longitud en la cual se ubica ...


Consulta de descubrimiento: https://www.datos.gov.co/resource/s54a-sgyg.csv?%24limit=10
Columnas reales: ['codigoestacion', 'codigosensor', 'fechaobservacion', 'valorobservado', 'nombreestacion', 'departamento', 'municipio', 'zonahidrografica', 'latitud', 'longitud', 'descripcionsensor', 'unidadmedida']
Tipos inferidos:
codigoestacion         int64
codigosensor           int64
fechaobservacion      object
valorobservado       float64
nombreestacion        object
departamento          object
municipio             object
zonahidrografica      object
latitud              float64
longitud             float64
descripcionsensor     object
unidadmedida          object


,codigoestacion,codigosensor,fechaobservacion,valorobservado,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,descripcionsensor,unidadmedida
0,21055501,240,2017-03-14T08:55:00.000,0.0,SIMON CAMPOS - AUT,HUILA,LA PLATA,ALTO MAGDALENA,2.345831,-75.878056,Precipitacion,mm
1,21206790,240,2005-10-16T05:50:00.000,0.0,HACIENDA SANTA ANA - AUT,CUNDINAMARCA,NEMOCÓN,ALTO MAGDALENA,5.090500,-73.881250,Precipitacion,mm
2,21015502,240,2018-12-15T07:20:00.000,0.0,SAN AGUSTIN - AUT,HUILA,SAN AGUSTÍN,ALTO MAGDALENA,1.851417,-76.304331,Precipitacion,mm
3,21237010,240,2008-10-26T14:00:00.000,0.0,NARINO - AUT,CUNDINAMARCA,NARIÑO,ALTO MAGDALENA,4.387778,-74.838375,Precipitacion,mm
4,21257120,240,2009-01-25T06:40:00.000,0.0,PUENTE NEGRO - AUT,TOLIMA,VILLAHERMOSA,ALTO MAGDALENA,4.940194,-75.089806,Precipitacion,mm


## 4. Conteos previos y consultas exactas

In [4]:
def where_period(start, end):
    return (
        f"fechaobservacion >= '{start}' "
        f"AND fechaobservacion < '{end}'"
    )


def count_period(start, end):
    params = {
        "$select": "count(*) as n",
        "$where": where_period(start, end),
    }
    response = get_retry(API_JSON, params)
    return int(response.json()[0]["n"]), response.url


rows_a_expected, count_url_a = count_period(PERIOD_A_START, PERIOD_A_END)
rows_b_expected, count_url_b = count_period(PERIOD_B_START, PERIOD_B_END)
print("Conteo período A:", rows_a_expected)
print("Consulta A:", count_url_a)
print("Conteo período B:", rows_b_expected)
print("Consulta B:", count_url_b)

Conteo período A: 139657
Consulta A: https://www.datos.gov.co/resource/s54a-sgyg.json?%24select=count%28%2A%29+as+n&%24where=fechaobservacion+%3E%3D+%272026-06-21T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272026-06-22T00%3A00%3A00%27
Conteo período B: 141007
Consulta B: https://www.datos.gov.co/resource/s54a-sgyg.json?%24select=count%28%2A%29+as+n&%24where=fechaobservacion+%3E%3D+%272026-06-22T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272026-06-23T00%3A00%3A00%27


## 5. Descarga paginada de los dos días

In [5]:
def count_rows_in_response(content):
    text = content.decode("utf-8-sig")
    return max(sum(1 for _ in csv.reader(io.StringIO(text, newline=""))) - 1, 0)


def download_period(start, end, expected_rows, target):
    target = Path(target)
    temporary = target.with_suffix(target.suffix + ".part")
    if temporary.exists():
        temporary.unlink()
    downloaded = 0
    offset = 0
    page_hashes = set()
    first_header = None
    urls = []
    with temporary.open("wb") as output:
        while downloaded < expected_rows:
            params = {
                "$where": where_period(start, end),
                "$order": ORDER,
                "$limit": PAGE_SIZE,
                "$offset": offset,
            }
            response = get_retry(API_CSV, params)
            urls.append(response.url)
            content = response.content
            header, body = content.split(b"\n", 1)
            header = header.rstrip(b"\r")
            if first_header is None:
                first_header = header
            elif header != first_header:
                raise RuntimeError("La cabecera cambió entre páginas.")
            page_hash = hashlib.sha256(content).hexdigest()
            if page_hash in page_hashes:
                raise RuntimeError("Se recibió una página duplicada.")
            page_hashes.add(page_hash)
            page_rows = count_rows_in_response(content)
            if page_rows == 0:
                raise RuntimeError("Página vacía antes de completar el conteo.")
            output.write(content if downloaded == 0 else body)
            downloaded += page_rows
            offset += page_rows
            print(f"{target.name}: {downloaded:,}/{expected_rows:,} filas")
            if page_rows < PAGE_SIZE:
                break
    if downloaded != expected_rows:
        raise RuntimeError(
            f"Conteo inconsistente: API={expected_rows}, archivo={downloaded}"
        )
    os.replace(temporary, target)
    return {
        "ruta": str(target),
        "filas": downloaded,
        "where": where_period(start, end),
        "order": ORDER,
        "limit": PAGE_SIZE,
        "primera_url": urls[0],
        "ultima_url": urls[-1],
        "paginas": len(urls),
    }


def existing_row_count(path):
    """Cuenta filas de datos de un CSV ya presente en disco."""
    with Path(path).open("r", encoding="utf-8-sig", newline="") as source:
        return max(sum(1 for _ in csv.reader(source)) - 1, 0)


def build_page_url(start, end, offset):
    """Reconstruye la URL exacta de una página de descarga."""
    params = {
        "$where": where_period(start, end),
        "$order": ORDER,
        "$limit": PAGE_SIZE,
        "$offset": offset,
    }
    return requests.Request("GET", API_CSV, params=params).prepare().url


def load_or_download(start, end, expected_rows, target):
    """Reutiliza la partición ya descargada si conserva el mismo conteo.

    La remedición en macOS debe realizarse sobre el mismo archivo y el mismo
    período del 22 de junio de 2026. Volver a descargar cambiaría S0 y haría
    incomparable el resultado con la medición anterior. Solo se descarga si el
    archivo no existe o si su conteo no coincide con el que reporta la API.
    """
    target = Path(target)
    if target.exists():
        rows_on_disk = existing_row_count(target)
        if rows_on_disk == expected_rows:
            pages = math.ceil(expected_rows / PAGE_SIZE)
            print(
                f"{target.name}: se reutiliza el archivo existente "
                f"({rows_on_disk:,} filas; coincide con el conteo de la API)."
            )
            return {
                "ruta": str(target),
                "filas": rows_on_disk,
                "where": where_period(start, end),
                "order": ORDER,
                "limit": PAGE_SIZE,
                "primera_url": build_page_url(start, end, 0),
                "ultima_url": build_page_url(
                    start, end, (pages - 1) * PAGE_SIZE
                ),
                "paginas": pages,
                "origen": (
                    "archivo preexistente reutilizado; no se volvió a descargar"
                ),
            }
        print(
            f"{target.name}: el archivo tiene {rows_on_disk:,} filas y la API "
            f"reporta {expected_rows:,}. Se descarga de nuevo."
        )
    result = download_period(start, end, expected_rows, target)
    result["origen"] = "descargado en esta ejecución"
    return result


download_a = load_or_download(
    PERIOD_A_START, PERIOD_A_END, rows_a_expected, COMPARISON_PATH
)
download_b = load_or_download(
    PERIOD_B_START, PERIOD_B_END, rows_b_expected, MAIN_PATH
)
print(json.dumps({"A": download_a, "B": download_b}, ensure_ascii=False, indent=2))


precipitacion_2026-06-21.csv: se reutiliza el archivo existente (139,657 filas; coincide con el conteo de la API).


precipitacion_2026-06-22.csv: se reutiliza el archivo existente (141,007 filas; coincide con el conteo de la API).
{
  "A": {
    "ruta": "/Users/juanpablocastro/Downloads/T1_Castro_JuanPablo/evidencia/precipitacion_2026-06-21.csv",
    "filas": 139657,
    "where": "fechaobservacion >= '2026-06-21T00:00:00' AND fechaobservacion < '2026-06-22T00:00:00'",
    "order": "fechaobservacion,codigoestacion,codigosensor,:id",
    "limit": 50000,
    "primera_url": "https://www.datos.gov.co/resource/s54a-sgyg.csv?%24where=fechaobservacion+%3E%3D+%272026-06-21T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272026-06-22T00%3A00%3A00%27&%24order=fechaobservacion%2Ccodigoestacion%2Ccodigosensor%2C%3Aid&%24limit=50000&%24offset=0",
    "ultima_url": "https://www.datos.gov.co/resource/s54a-sgyg.csv?%24where=fechaobservacion+%3E%3D+%272026-06-21T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272026-06-22T00%3A00%3A00%27&%24order=fechaobservacion%2Ccodigoestacion%2Ccodigosensor%2C%3Aid&%24limit=50000&%24offset=1

## 6. Tamaño real en disco \(S_0\), hash y validación

En esta entrega, \(S_0\) es el tamaño de **una partición diaria** (22 de junio de 2026), no el tamaño acumulado de la fuente ni del repositorio histórico. El horizonte calculado más adelante describe el crecimiento hipotético de una partición diaria representativa. Un repositorio que agrega nuevas particiones diariamente crece por acumulación y requiere otro modelo de capacidad.

In [6]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        while block := source.read(4 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


size_a_bytes = os.path.getsize(COMPARISON_PATH)
size_b_bytes = os.path.getsize(MAIN_PATH)
S0_bytes = size_b_bytes
S0_mb = S0_bytes / BYTES_PER_MB
S0_gb = S0_bytes / BYTES_PER_GB
hash_a = sha256_file(COMPARISON_PATH)
hash_b = sha256_file(MAIN_PATH)
disk_summary = {
    "S0_bytes": S0_bytes,
    "S0_MB_1024": S0_mb,
    "S0_GB_1024": S0_gb,
    "alcance_S0": (
        "tamaño de la partición diaria 2026-06-22; "
        "no es el acumulado histórico"
    ),
    "filas": rows_b_expected,
    "sha256": hash_b,
    "ruta": str(MAIN_PATH),
}
print(json.dumps(disk_summary, ensure_ascii=False, indent=2))

{
  "S0_bytes": 21953076,
  "S0_MB_1024": 20.936084747314453,
  "S0_GB_1024": 0.02044539526104927,
  "alcance_S0": "tamaño de la partición diaria 2026-06-22; no es el acumulado histórico",
  "filas": 141007,
  "sha256": "9a8dc75af1969e21ad7e13bddd9fad0291ebbeba2a0b1418cd4237f81a5155be",
  "ruta": "/Users/juanpablocastro/Downloads/T1_Castro_JuanPablo/evidencia/precipitacion_2026-06-22.csv"
}


## 7. Memoria útil \(M\) y programas abiertos

In [7]:
memory_at = datetime.now(LOCAL_TZ)
vm_before = psutil.virtual_memory()
M_bytes = vm_before.available
M_gb = M_bytes / BYTES_PER_GB

processes = []
for process in psutil.process_iter(["pid", "name", "memory_info"]):
    try:
        info = process.info
        # En macOS psutil devuelve memory_info = None para los procesos que
        # el usuario no tiene permiso de inspeccionar. Se omiten en lugar de
        # interrumpir la medición; M no depende de este inventario.
        memory_info = info.get("memory_info")
        if memory_info is None:
            continue
        processes.append({
            "pid": info["pid"],
            "programa": info["name"],
            "rss_bytes": memory_info.rss,
        })
    except (psutil.NoSuchProcess, psutil.AccessDenied, AttributeError):
        pass
top_processes = (
    pd.DataFrame(processes)
    .sort_values("rss_bytes", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_processes["rss_MB_1024"] = top_processes["rss_bytes"] / BYTES_PER_MB
top_processes.to_csv(
    EVIDENCE_DIR / "procesos_memoria.csv", index=False, encoding="utf-8-sig"
)
memory_summary = {
    "fecha_hora_local": memory_at.isoformat(),
    "RAM_total_bytes": vm_before.total,
    "RAM_disponible_M_bytes": M_bytes,
    "RAM_disponible_M_GB_1024": M_gb,
    "porcentaje_uso": vm_before.percent,
}
print(json.dumps(memory_summary, ensure_ascii=False, indent=2))
display(top_processes)

{
  "fecha_hora_local": "2026-07-29T15:17:10.825578-05:00",
  "RAM_total_bytes": 8589934592,
  "RAM_disponible_M_bytes": 1609220096,
  "RAM_disponible_M_GB_1024": 1.4987030029296875,
  "porcentaje_uso": 81.3
}


,pid,programa,rss_bytes,rss_MB_1024
0,4152,Claude Helper (Renderer),543014912,517.859375
1,4038,Claude,204881920,195.390625
2,7173,claude,193609728,184.640625
3,5929,claude,157908992,150.593750
4,9185,python3.12,137625600,131.250000
5,9183,python3.12,90013696,85.843750
6,7875,Code Helper (Renderer),89620480,85.468750
7,7860,Code,84721664,80.796875
8,3933,Spotify,79495168,75.812500
9,8916,Code Helper (Renderer),68321280,65.156250


## 8. Carga con pandas y factor de expansión \(k\)

In [8]:
df = pd.read_csv(MAIN_PATH, low_memory=False)
memory_dataframe_bytes = df.memory_usage(deep=True).sum()
memory_dataframe_mb = memory_dataframe_bytes / BYTES_PER_MB
memory_dataframe_gb = memory_dataframe_bytes / BYTES_PER_GB
k = memory_dataframe_bytes / S0_bytes
if k <= 0:
    raise RuntimeError("El factor k no es positivo.")

text_columns = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
text_ratio = len(text_columns) / df.shape[1]
dataframe_summary = {
    "filas": int(df.shape[0]),
    "columnas": int(df.shape[1]),
    "tipos": {column: str(dtype) for column, dtype in df.dtypes.items()},
    "columnas_texto": text_columns,
    "cantidad_columnas_texto": len(text_columns),
    "proporcion_columnas_texto": text_ratio,
    "memoria_dataframe_bytes": int(memory_dataframe_bytes),
    "memoria_dataframe_MB_1024": memory_dataframe_mb,
    "memoria_dataframe_GB_1024": memory_dataframe_gb,
    "k": k,
}
print(json.dumps(dataframe_summary, ensure_ascii=False, indent=2))
display(df.head())

{
  "filas": 141007,
  "columnas": 12,
  "tipos": {
    "codigoestacion": "int64",
    "codigosensor": "int64",
    "fechaobservacion": "object",
    "valorobservado": "float64",
    "nombreestacion": "object",
    "departamento": "object",
    "municipio": "object",
    "zonahidrografica": "object",
    "latitud": "float64",
    "longitud": "float64",
    "descripcionsensor": "object",
    "unidadmedida": "object"
  },
  "columnas_texto": [
    "fechaobservacion",
    "nombreestacion",
    "departamento",
    "municipio",
    "zonahidrografica",
    "descripcionsensor",
    "unidadmedida"
  ],
  "cantidad_columnas_texto": 7,
  "proporcion_columnas_texto": 0.5833333333333334,
  "memoria_dataframe_bytes": 71118728,
  "memoria_dataframe_MB_1024": 67.82410430908203,
  "memoria_dataframe_GB_1024": 0.06623447686433792,
  "k": 3.239579182434389
}


,codigoestacion,codigosensor,fechaobservacion,valorobservado,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,descripcionsensor,unidadmedida
0,11027030,240,2026-06-22T00:00:00.000,0.0,EL SIETE,CHOCO,EL CARMEN,ATRATO - DARIÉN,5.862000,-76.152056,PRECIPITACIÓN,mm
1,11030010,240,2026-06-22T00:00:00.000,0.0,CERTEGUI,CHOCO,CÉRTEGUI,ATRATO - DARIÉN,5.380000,-76.610000,PRECIPITACIÓN,mm
2,11035010,240,2026-06-22T00:00:00.000,0.0,LLORO,CHOCO,LLORÓ,ATRATO - DARIÉN,5.499000,-76.539000,PRECIPITACIÓN,mm
3,11035030,240,2026-06-22T00:00:00.000,0.6,UNION PANAMERICANA,CHOCO,UNIÓN PANAMERICANA,ATRATO - DARIÉN,5.284828,-76.627822,PRECIPITACIÓN,mm
4,11045010,240,2026-06-22T00:00:00.000,0.0,AEROPUERTO EL CARAÑO,CHOCO,QUIBDÓ,ATRATO - DARIÉN,5.690556,-76.643778,PRECIPITACIÓN,mm


## 9. Clave candidata: estación + sensor + fecha

In [9]:
candidate_key = ["codigoestacion", "codigosensor", "fechaobservacion"]
missing_key_columns = [column for column in candidate_key if column not in df.columns]
if missing_key_columns:
    raise KeyError(f"Faltan columnas de clave: {missing_key_columns}")

key_nulls = df[candidate_key].isna().sum()
duplicate_mask = df.duplicated(subset=candidate_key, keep=False)
duplicate_rows = int(duplicate_mask.sum())
unique_combinations = int(df[candidate_key].drop_duplicates().shape[0])
duplicate_percentage = duplicate_rows / len(df) * 100
key_summary = {
    "clave_candidata": candidate_key,
    "filas": len(df),
    "combinaciones_unicas": unique_combinations,
    "filas_en_grupos_duplicados": duplicate_rows,
    "porcentaje_filas_duplicadas": duplicate_percentage,
    "nulos": {column: int(value) for column, value in key_nulls.items()},
    "es_clave_unica_en_el_periodo": duplicate_rows == 0,
}
print(json.dumps(key_summary, ensure_ascii=False, indent=2))
pd.DataFrame([key_summary]).to_csv(
    EVIDENCE_DIR / "verificacion_clave.csv", index=False, encoding="utf-8-sig"
)

{
  "clave_candidata": [
    "codigoestacion",
    "codigosensor",
    "fechaobservacion"
  ],
  "filas": 141007,
  "combinaciones_unicas": 141007,
  "filas_en_grupos_duplicados": 0,
  "porcentaje_filas_duplicadas": 0.0,
  "nulos": {
    "codigoestacion": 0,
    "codigosensor": 0,
    "fechaobservacion": 0
  },
  "es_clave_unica_en_el_periodo": true
}


## 10. Frecuencia declarada y frecuencia observada

In [10]:
frequency_df = df[candidate_key].copy()
frequency_df["fechaobservacion"] = pd.to_datetime(
    frequency_df["fechaobservacion"], errors="coerce"
)
frequency_df = frequency_df.dropna(subset=["fechaobservacion"]).sort_values(candidate_key)
intervals = (
    frequency_df.groupby(["codigoestacion", "codigosensor"], dropna=False)
    ["fechaobservacion"].diff().dropna()
)
positive_intervals = intervals[intervals > pd.Timedelta(0)]
mode_interval = positive_intervals.mode().iloc[0]
median_interval = positive_intervals.median()
mode_share = float((positive_intervals == mode_interval).mean())
top_intervals = positive_intervals.value_counts().head(8)
frequency_table = pd.DataFrame({
    "intervalo": top_intervals.index.astype(str),
    "conteo": top_intervals.values,
    "proporcion": top_intervals.values / len(positive_intervals),
})
frequency_table.to_csv(
    EVIDENCE_DIR / "frecuencia_observada.csv", index=False, encoding="utf-8-sig"
)
frequency_summary = {
    "frecuencia_actualizacion_declarada": custom_data.get("Frecuencia de Actualización"),
    "cadencia_mencionada_en_descripcion": "cada 10 minutos",
    "moda_observada": str(mode_interval),
    "mediana_observada": str(median_interval),
    "proporcion_intervalo_modal": mode_share,
}
print(json.dumps(frequency_summary, ensure_ascii=False, indent=2))
display(frequency_table)

{
  "frecuencia_actualizacion_declarada": "Diaria",
  "cadencia_mencionada_en_descripcion": "cada 10 minutos",
  "moda_observada": "0 days 00:01:00",
  "mediana_observada": "0 days 00:02:00",
  "proporcion_intervalo_modal": 0.44164655460868696
}


,intervalo,conteo,proporcion
0,0 days 00:01:00,62035,0.441647
1,0 days 00:10:00,60336,0.429551
2,0 days 00:02:00,14825,0.105544
3,0 days 00:03:00,835,0.005945
4,0 days 01:00:00,583,0.004151
5,0 days 00:04:00,557,0.003965
6,0 days 00:05:00,404,0.002876
7,0 days 00:06:00,174,0.001239


## 11. Comparación del esquema entre los dos períodos

In [11]:
df_a = pd.read_csv(COMPARISON_PATH, low_memory=False)
all_columns = list(dict.fromkeys(df_a.columns.tolist() + df.columns.tolist()))
schema_rows = []
for column in all_columns:
    in_a = column in df_a.columns
    in_b = column in df.columns
    schema_rows.append({
        "columna": column,
        "existe_A": in_a,
        "existe_B": in_b,
        "posicion_A": df_a.columns.get_loc(column) if in_a else np.nan,
        "posicion_B": df.columns.get_loc(column) if in_b else np.nan,
        "tipo_A": str(df_a[column].dtype) if in_a else "",
        "tipo_B": str(df[column].dtype) if in_b else "",
        "proporcion_nulos_A": float(df_a[column].isna().mean()) if in_a else np.nan,
        "proporcion_nulos_B": float(df[column].isna().mean()) if in_b else np.nan,
    })
schema_comparison = pd.DataFrame(schema_rows)
schema_comparison["mismo_tipo"] = schema_comparison["tipo_A"] == schema_comparison["tipo_B"]
schema_comparison["misma_posicion"] = schema_comparison["posicion_A"] == schema_comparison["posicion_B"]
schema_identical = (
    df_a.columns.tolist() == df.columns.tolist()
    and schema_comparison["mismo_tipo"].all()
)
schema_comparison.to_csv(
    EVIDENCE_DIR / "comparacion_esquema.csv", index=False, encoding="utf-8-sig"
)
print("Nombres, orden y tipos inferidos idénticos:", bool(schema_identical))
display(schema_comparison)

Nombres, orden y tipos inferidos idénticos: True


,columna,existe_A,existe_B,posicion_A,posicion_B,tipo_A,tipo_B,proporcion_nulos_A,proporcion_nulos_B,mismo_tipo,misma_posicion
0,codigoestacion,True,True,0,0,int64,int64,0.0,0.0,True,True
1,codigosensor,True,True,1,1,int64,int64,0.0,0.0,True,True
2,fechaobservacion,True,True,2,2,object,object,0.0,0.0,True,True
3,valorobservado,True,True,3,3,float64,float64,0.0,0.0,True,True
4,nombreestacion,True,True,4,4,object,object,0.0,0.0,True,True
5,departamento,True,True,5,5,object,object,0.0,0.0,True,True
6,municipio,True,True,6,6,object,object,0.0,0.0,True,True
7,zonahidrografica,True,True,7,7,object,object,0.0,0.0,True,True
8,latitud,True,True,8,8,float64,float64,0.0,0.0,True,True
9,longitud,True,True,9,9,float64,float64,0.0,0.0,True,True


## 12. Ausencia de datos personales identificables

In [12]:
personal_terms = [
    "cédula", "cedula", "documento", "teléfono", "telefono",
    "correo", "email", "persona", "ciudadano",
]
pii_hits = []
for _, row in metadata_columns.fillna("").iterrows():
    text = " ".join(str(value) for value in row).lower()
    hits = [term for term in personal_terms if term in text]
    if hits:
        pii_hits.append({"columna": row["nombre_api"], "terminos": hits})
pii_summary = {
    "coincidencias_en_nombres_y_descripciones": pii_hits,
    "conclusion": (
        "Las columnas describen estaciones, sensores, fechas, ubicación y "
        "mediciones técnicas. No se identificaron nombres de ciudadanos, "
        "documentos, teléfonos ni correos. 'nombreestacion' es el nombre "
        "de una estación meteorológica, no de una persona."
    ),
}
print(json.dumps(pii_summary, ensure_ascii=False, indent=2))

{
  "coincidencias_en_nombres_y_descripciones": [],
  "conclusion": "Las columnas describen estaciones, sensores, fechas, ubicación y mediciones técnicas. No se identificaron nombres de ciudadanos, documentos, teléfonos ni correos. 'nombreestacion' es el nombre de una estación meteorológica, no de una persona."
}


## 13. Estimación histórica de la tasa de crecimiento \(g\)

La evidencia histórica principal compara junio de 2025 con junio de 2026 mediante `$select=count(*)`. Son el mismo mes calendario, separados por un año. No se descargan esos meses completos. La comparación diaria anterior se conserva únicamente como escenario operativo de corto plazo.

In [13]:
HIST_A_START = "2025-06-01T00:00:00"
HIST_A_END = "2025-07-01T00:00:00"
HIST_B_START = "2026-06-01T00:00:00"
HIST_B_END = "2026-07-01T00:00:00"

historical_rows_a, historical_count_url_a = count_period(
    HIST_A_START, HIST_A_END
)
historical_rows_b, historical_count_url_b = count_period(
    HIST_B_START, HIST_B_END
)
historical_periods_years = 1
historical_periods_months = 12
g_historical_annual = (
    historical_rows_b / historical_rows_a
) ** (1 / historical_periods_years) - 1
g_historical_monthly = (
    historical_rows_b / historical_rows_a
) ** (1 / historical_periods_months) - 1

# La tasa de dos días se conserva solo como contraste de corto plazo.
g_short_term_daily_size = size_b_bytes / size_a_bytes - 1
g_short_term_daily_records = rows_b_expected / rows_a_expected - 1

growth_summary = {
    "metodo_definitivo": (
        "conteos API del mismo mes calendario en años consecutivos"
    ),
    "periodo_historico_A": [HIST_A_START, HIST_A_END],
    "periodo_historico_B": [HIST_B_START, HIST_B_END],
    "registros_historico_A": historical_rows_a,
    "registros_historico_B": historical_rows_b,
    "periodos_transcurridos": historical_periods_years,
    "unidad_tasa_principal": "año",
    "formula": "(rows_B / rows_A) ** (1 / n_años) - 1",
    "g_historico_anual": g_historical_annual,
    "g_historico_mensual_equivalente": g_historical_monthly,
    "consulta_conteo_A": historical_count_url_a,
    "consulta_conteo_B": historical_count_url_b,
    "escenario_corto_plazo": {
        "periodo_A": [PERIOD_A_START, PERIOD_A_END],
        "periodo_B": [PERIOD_B_START, PERIOD_B_END],
        "g_tamano_diario": g_short_term_daily_size,
        "g_registros_diario": g_short_term_daily_records,
        "uso": "escenario operativo; no tendencia histórica",
    },
}
print(json.dumps(growth_summary, ensure_ascii=False, indent=2))

growth_evidence = pd.DataFrame([
    {
        "periodo": "junio 2025",
        "inicio": HIST_A_START,
        "fin_exclusivo": HIST_A_END,
        "registros": historical_rows_a,
        "periodos_transcurridos": historical_periods_years,
        "unidad": "año",
        "g_historico_anual": g_historical_annual,
        "g_mensual_equivalente": g_historical_monthly,
        "consulta": historical_count_url_a,
    },
    {
        "periodo": "junio 2026",
        "inicio": HIST_B_START,
        "fin_exclusivo": HIST_B_END,
        "registros": historical_rows_b,
        "periodos_transcurridos": historical_periods_years,
        "unidad": "año",
        "g_historico_anual": g_historical_annual,
        "g_mensual_equivalente": g_historical_monthly,
        "consulta": historical_count_url_b,
    },
])
growth_evidence.to_csv(
    EVIDENCE_DIR / "estimacion_crecimiento.csv",
    index=False,
    encoding="utf-8-sig",
)
display(growth_evidence)

{
  "metodo_definitivo": "conteos API del mismo mes calendario en años consecutivos",
  "periodo_historico_A": [
    "2025-06-01T00:00:00",
    "2025-07-01T00:00:00"
  ],
  "periodo_historico_B": [
    "2026-06-01T00:00:00",
    "2026-07-01T00:00:00"
  ],
  "registros_historico_A": 3120484,
  "registros_historico_B": 2971298,
  "periodos_transcurridos": 1,
  "unidad_tasa_principal": "año",
  "formula": "(rows_B / rows_A) ** (1 / n_años) - 1",
  "g_historico_anual": -0.04780860917729424,
  "g_historico_mensual_equivalente": -0.00407411349065856,
  "consulta_conteo_A": "https://www.datos.gov.co/resource/s54a-sgyg.json?%24select=count%28%2A%29+as+n&%24where=fechaobservacion+%3E%3D+%272025-06-01T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272025-07-01T00%3A00%3A00%27",
  "consulta_conteo_B": "https://www.datos.gov.co/resource/s54a-sgyg.json?%24select=count%28%2A%29+as+n&%24where=fechaobservacion+%3E%3D+%272026-06-01T00%3A00%3A00%27+AND+fechaobservacion+%3C+%272026-07-01T00%3A00%3A00%27",
  "

,periodo,inicio,fin_exclusivo,registros,periodos_transcurridos,unidad,g_historico_anual,g_mensual_equivalente,consulta
0,junio 2025,2025-06-01T00:00:00,2025-07-01T00:00:00,3120484,1,año,-0.047809,-0.004074,https://www.datos.gov.co/resource/s54a-sgyg.js...
1,junio 2026,2026-06-01T00:00:00,2026-07-01T00:00:00,2971298,1,año,-0.047809,-0.004074,https://www.datos.gov.co/resource/s54a-sgyg.js...


## 14. Horizonte de saturación e interpretación de \(S_0\)

Como \(g\) histórico es negativo, mantener esa tasa reduciría el tamaño de una partición representativa y no produciría saturación futura. No se fuerza la fórmula con una tasa histórica negativa.

Para la sensibilidad se usan **las mediciones propias de esta ejecución en macOS**: \(M\) obtenida con `psutil.virtual_memory().available`, \(k\) medido con `df.memory_usage(deep=True).sum()` y \(S_0\) medido con `os.path.getsize()`, todos sobre esta MacBook. La tasa de referencia es \(g=0,01\) anual. \(M\) y \(S_0\) están en gigabytes (`1024 ** 3`), \(k\) es adimensional y el resultado queda en años.

El horizonte se refiere a **una partición diaria cuyo tamaño aumenta**. No describe el tamaño total de un repositorio que incorpora una partición nueva cada día.


In [14]:
def compute_threshold_periods(M_value, k_value, S0_value, growth_rate):
    if growth_rate <= 0:
        raise ValueError("La tasa debe ser positiva para un horizonte futuro.")
    ratio_value = M_value / (k_value * S0_value)
    if ratio_value <= 0:
        raise ValueError("M / (k * S0) debe ser positivo.")
    return math.log(ratio_value) / math.log(1 + growth_rate)


historical_t_umbral_years = None
historical_interpretation = (
    "No existe horizonte futuro de saturación bajo la tasa histórica "
    "negativa; una partición representativa disminuiría en el modelo."
)

# Mediciones propias de esta ejecución en macOS (MacBook Neo).
# S0 y k provienen del mismo archivo del 22 de junio de 2026; M es la memoria
# disponible medida inmediatamente antes de cargar el CSV.
M_sensitivity_gb = M_bytes / BYTES_PER_GB
k_sensitivity = k
S0_sensitivity_gb = S0_gb
reference_rate = 0.01

reference_t_years = compute_threshold_periods(
    M_sensitivity_gb,
    k_sensitivity,
    S0_sensitivity_gb,
    reference_rate,
)

sensitivity_rows = []
for scenario_rate in [0.01, 0.02, 0.05, 0.10]:
    scenario_t_years = compute_threshold_periods(
        M_sensitivity_gb,
        k_sensitivity,
        S0_sensitivity_gb,
        scenario_rate,
    )
    sensitivity_rows.append({
        "escenario": f"{scenario_rate:.0%} anual",
        "M_GB": M_sensitivity_gb,
        "k": k_sensitivity,
        "S0_GB": S0_sensitivity_gb,
        "g_anual": scenario_rate,
        "t_umbral_anios": scenario_t_years,
        "alcance": "crecimiento del tamaño de una partición diaria",
    })

# La comparación de dos días queda separada de la tendencia histórica.
short_term_t_days = compute_threshold_periods(
    M_bytes, k, S0_bytes, g_short_term_daily_size
)
sensitivity_rows.append({
    "escenario": "variación observada entre dos días",
    "M_GB": M_bytes / BYTES_PER_GB,
    "k": k,
    "S0_GB": S0_bytes / BYTES_PER_GB,
    "g_anual": np.nan,
    "t_umbral_anios": np.nan,
    "alcance": (
        f"escenario de corto plazo: g={g_short_term_daily_size:.12f} "
        f"por día; t={short_term_t_days:.6f} días"
    ),
})
sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df.to_csv(
    EVIDENCE_DIR / "sensibilidad_umbral.csv",
    index=False,
    encoding="utf-8-sig",
)

threshold_summary = {
    "M_medida_en_esta_ejecucion_GB": M_bytes / BYTES_PER_GB,
    "M_sensibilidad_GB": M_sensitivity_gb,
    "k_sensibilidad": k_sensitivity,
    "S0_sensibilidad_GB": S0_sensitivity_gb,
    "alcance_S0": "partición diaria; no acumulado histórico",
    "g_historico_anual": g_historical_annual,
    "t_umbral_historico_anios": historical_t_umbral_years,
    "interpretacion_historica": historical_interpretation,
    "formula_escenarios": "log(M / (k * S0)) / log(1 + g_escenario)",
    "escenario_referencia": "1 % anual",
    "sustitucion_referencia": (
        f"log({M_sensitivity_gb} / "
        f"({k_sensitivity} * {S0_sensitivity_gb})) / "
        f"log(1 + {reference_rate})"
    ),
    "t_umbral_referencia_anios": reference_t_years,
    "nota_remedicion": (
        "Mediciones definitivas realizadas en macOS (MacBook Neo). "
        "Reemplazan la línea base previa de Windows "
        "(M=7,431983948 GB, k=3,2395791824, t_1%=474,390960 años). "
        "El período, el archivo, la clave candidata, la licencia y el "
        "método de cálculo de g no cambian."
    ),
    "escenario_corto_plazo_dias": short_term_t_days,
}
print(json.dumps(threshold_summary, ensure_ascii=False, indent=2))
display(sensitivity_df)

{
  "M_medida_en_esta_ejecucion_GB": 1.4987030029296875,
  "M_sensibilidad_GB": 1.4987030029296875,
  "k_sensibilidad": 3.239579182434389,
  "S0_sensibilidad_GB": 0.02044539526104927,
  "alcance_S0": "partición diaria; no acumulado histórico",
  "g_historico_anual": -0.04780860917729424,
  "t_umbral_historico_anios": null,
  "interpretacion_historica": "No existe horizonte futuro de saturación bajo la tasa histórica negativa; una partición representativa disminuiría en el modelo.",
  "formula_escenarios": "log(M / (k * S0)) / log(1 + g_escenario)",
  "escenario_referencia": "1 % anual",
  "sustitucion_referencia": "log(1.4987030029296875 / (3.239579182434389 * 0.02044539526104927)) / log(1 + 0.01)",
  "t_umbral_referencia_anios": 313.4724129716826,
  "nota_remedicion": "Mediciones definitivas realizadas en macOS (MacBook Neo). Reemplazan la línea base previa de Windows (M=7,431983948 GB, k=3,2395791824, t_1%=474,390960 años). El período, el archivo, la clave candidata, la licencia y el

,escenario,M_GB,k,S0_GB,g_anual,t_umbral_anios,alcance
0,1% anual,1.498703,3.239579,0.020445,0.01,313.472413,crecimiento del tamaño de una partición diaria
1,2% anual,1.498703,3.239579,0.020445,0.02,157.512141,crecimiento del tamaño de una partición diaria
2,5% anual,1.498703,3.239579,0.020445,0.05,63.929980,crecimiento del tamaño de una partición diaria
3,10% anual,1.498703,3.239579,0.020445,0.10,32.726349,crecimiento del tamaño de una partición diaria
4,variación observada entre dos días,1.498703,3.239579,0.020445,NaN,NaN,escenario de corto plazo: g=0.008763988780 por...


## 15. Resumen final y evidencias

In [15]:
final_summary = pd.DataFrame([
    ["S0 diario", S0_bytes, "bytes"],
    ["Memoria DataFrame", memory_dataframe_bytes, "bytes"],
    ["k", k, "adimensional"],
    ["M", M_bytes, "bytes disponibles"],
    ["g histórico", g_historical_annual, "por año"],
    ["t histórico", np.nan, "no existe horizonte futuro con g < 0"],
    ["t escenario 1 %", reference_t_years, "años"],
    ["t corto plazo", short_term_t_days, "días; no tendencia histórica"],
], columns=["metrica", "valor", "unidad"])
display(final_summary)

results = {
    "dataset": {
        "id": DATASET_ID,
        "portal_url": PORTAL_URL,
        "api_csv": API_CSV,
        "metadata_url": METADATA_URL,
        "entidad": metadata.get("attribution"),
        "descripcion": metadata.get("description"),
        "licencia": metadata.get("license"),
        "frecuencia_actualizacion": custom_data.get("Frecuencia de Actualización"),
    },
    "entorno": environment,
    "periodos_medicion": {
        "principal": [PERIOD_B_START, PERIOD_B_END],
        "comparacion_diaria": [PERIOD_A_START, PERIOD_A_END],
        "conteo_principal": rows_b_expected,
        "conteo_comparacion_diaria": rows_a_expected,
    },
    "consultas": {
        "descubrimiento": discovery_response.url,
        "conteo_diario_A": count_url_a,
        "conteo_diario_B": count_url_b,
        "descarga_diaria_A": download_a,
        "descarga_diaria_B": download_b,
        "conteo_historico_A": historical_count_url_a,
        "conteo_historico_B": historical_count_url_b,
    },
    "disco": disk_summary,
    "memoria_equipo": memory_summary,
    "dataframe": dataframe_summary,
    "clave": key_summary,
    "frecuencia": frequency_summary,
    "esquema_identico": bool(schema_identical),
    "datos_personales": pii_summary,
    "crecimiento": growth_summary,
    "umbral": threshold_summary,
    "sensibilidad": sensitivity_df.replace({np.nan: None}).to_dict(
        orient="records"
    ),
}
(EVIDENCE_DIR / "resultados_medicion.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

query_text = f'''Endpoint: {API_CSV}
Período principal diario: [{PERIOD_B_START}, {PERIOD_B_END})
Conteo previo: {rows_b_expected}
$where={download_b["where"]}
$order={download_b["order"]}
$limit={download_b["limit"]}
$offset=0, 50000, ... hasta completar
Primera URL: {download_b["primera_url"]}
Última URL: {download_b["ultima_url"]}
Tamaño bytes: {size_b_bytes}
SHA-256: {hash_b}

Período comparable diario: [{PERIOD_A_START}, {PERIOD_A_END})
Conteo previo: {rows_a_expected}
Primera URL: {download_a["primera_url"]}
Última URL: {download_a["ultima_url"]}
Tamaño bytes: {size_a_bytes}
SHA-256: {hash_a}

Estimación histórica por conteos:
Junio 2025: {historical_rows_a} registros
Consulta: {historical_count_url_a}
Junio 2026: {historical_rows_b} registros
Consulta: {historical_count_url_b}
Períodos transcurridos: 1 año (12 meses)
g anual: {g_historical_annual}
g mensual equivalente: {g_historical_monthly}
'''
(EVIDENCE_DIR / "consulta_descarga.txt").write_text(
    query_text, encoding="utf-8"
)

readme = '''# Evidencias

- `metadata_dataset.json`: metadatos oficiales de Socrata.
- `precipitacion_2026-06-22.csv`: partición diaria principal cruda.
- `precipitacion_2026-06-21.csv`: partición diaria comparable cruda.
- `consulta_descarga.txt`: consultas diarias y conteos históricos mensuales.
- `resultados_medicion.json`: cifras estructuradas usadas para la ficha.
- `procesos_memoria.csv`: programas con mayor memoria RSS al medir M.
- `verificacion_clave.csv`: nulos y duplicados de la clave candidata.
- `frecuencia_observada.csv`: intervalos temporales más frecuentes.
- `comparacion_esquema.csv`: comparación de columnas, orden, tipos y nulos.
- `estimacion_crecimiento.csv`: conteos de junio de 2025 y junio de 2026.
- `sensibilidad_umbral.csv`: horizontes para escenarios positivos.

`S0` corresponde al tamaño de una partición diaria. Los archivos mensuales
no se descargaron: la API permitió obtener sus conteos con `count(*)`.
'''
(EVIDENCE_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Evidencias actualizadas en", EVIDENCE_DIR)

,metrica,valor,unidad
0,S0 diario,2.195308e+07,bytes
1,Memoria DataFrame,7.111873e+07,bytes
2,k,3.239579e+00,adimensional
3,M,1.609220e+09,bytes disponibles
4,g histórico,-4.780861e-02,por año
5,t histórico,NaN,no existe horizonte futuro con g < 0
6,t escenario 1 %,3.134724e+02,años
7,t corto plazo,3.574631e+02,días; no tendencia histórica


Evidencias actualizadas en /Users/juanpablocastro/Downloads/T1_Castro_JuanPablo/evidencia


## 16. Conclusiones, limitaciones e implicaciones para el pipeline

La evidencia histórica principal compara junio de 2025 (3.120.484 registros) con junio de 2026 (2.971.298 registros) y arroja \(g=-4,780861\%\) anual. Ambos conteos fueron reconfirmados en vivo contra la API en esta ejecución. Una tasa histórica negativa no produce un horizonte futuro de saturación.

El escenario de sensibilidad de 1 % anual usa las mediciones propias de esta MacBook: \(M=1,498703002930\) GB, \(k=3,239579182434\) y \(S_0=0,020445395261\) GB. Su horizonte es aproximadamente 313,47 años. No es una tasa histórica ni una predicción.

\(S_0\), la memoria del DataFrame y \(k\) coinciden con la medición anterior en Windows, y eso es el resultado esperado: el archivo es byte a byte el mismo (hash SHA-256 verificado) y se usó la misma versión de pandas, por lo que `memory_usage(deep=True)` es determinista. La única magnitud que depende de la máquina es \(M\), que bajó de 7,431983948 GB a 1,498703002930 GB y explica por sí sola que el horizonte pasara de 474,39 a 313,47 años.

\(S_0\) es una partición diaria. El cálculo responde a la pregunta: “¿cuándo una partición diaria representativa, expandida en pandas, alcanzaría la memoria disponible si aumentara a una tasa anual dada?”. No responde cuándo se llenará el repositorio completo.

Un pipeline incremental agrega particiones diarias. Su almacenamiento acumulado crece principalmente por la suma de particiones, aunque el tamaño medio de cada partición cambie. Para dimensionar el repositorio completo se necesitarían el tamaño acumulado actual, la retención y un modelo aditivo; esos elementos no se midieron aquí.
